# Question 6: Structured vs Unstructured Output

Use the exact homework Pydantic model with OpenAI structured outputs. Compare API-reported input tokens against the unstructured Question 5 version using the same documents, search, instructions, prompt, and `gpt-4o-mini` model.

Run the cells in order in the project environment. Credentials are loaded from the environment or repository `.env`. The comparison cell makes **two paid API requests**. No token counts are estimated.

In [ ]:
from pathlib import Path
from gitsource import chunk_documents
from minsearch import Index

# Support running from the repository root or homework/.
project_root = Path.cwd()
if not (project_root / "books_text").is_dir():
    project_root = project_root.parent

books_text_dir = project_root / "books_text"
markdown_files = sorted(books_text_dir.glob("*.md"))
if not markdown_files:
    raise FileNotFoundError(f"No Markdown books found in {books_text_dir}")

## Load and chunk all books

Keep non-empty lines and apply the same 100-line window and 50-line step as Question 2.

In [ ]:
books = []
for markdown_path in markdown_files:
    lines = markdown_path.read_text(encoding="utf-8").splitlines()
    books.append({
        "source": markdown_path.name,
        "content": [line for line in lines if line.strip()],
    })

chunks = chunk_documents(books, size=100, step=50)
print(f"Created {len(chunks):,} chunks from {len(books)} books.")

## Prepare searchable documents

MinSearch expects text strings. Join each chunk’s lines with newlines, preserving its source and starting position without modifying the original chunk.

In [ ]:
def prepare_documents(chunks):
    documents = []
    for chunk in chunks:
        document = chunk.copy()
        document["content"] = "\n".join(chunk["content"])
        documents.append(document)
    return documents


documents = prepare_documents(chunks)

## Build the search index

In [ ]:
index = Index(text_fields=["content"])
index.fit(documents)

print(f"Number of documents (chunks) indexed: {len(documents)}")

## Initialize the OpenAI client

Load the repository’s `.env` without overriding existing environment variables.

In [ ]:
import json
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(project_root / ".env", override=False)
if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY in your environment or the repository's .env file.")

openai_client = OpenAI()

## Define the RAG functions

Keep the homework’s instructions, JSON context, prompt template, five search results, and default model. Extend the return values to include actual token usage reported by OpenAI.

In [ ]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()


def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt


def search(question):
    return index.search(question, num_results=5)


def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return {
        "answer": response.output_text,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }


def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## Preserve the Question 5 baseline

Keep the original functions before replacing `llm` and `rag` with structured versions.

In [ ]:
llm_unstructured = llm


def rag_unstructured(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_unstructured(prompt, instructions)

## Define the response model

Use the supplied fields and descriptions unchanged so the schema is consistent with the homework.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal


class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")

## Modify `llm` and `rag` for structured outputs

`responses.parse` submits the Pydantic schema and returns a parsed model. Token usage comes from the same API response.

In [ ]:
def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=RAGResponse,
    )

    if response.output_parsed is None:
        raise RuntimeError(
            "No structured answer was returned. Check for a refusal or incomplete response."
        )

    return {
        "answer": response.output_parsed,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }


def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)

## Run both versions and compare

The unstructured call reproduces Question 5 locally. Subtract its input token count from the structured count to answer the homework. Compare the printed baseline with your saved Q5 count as well; changed book contents or package versions can affect it.

In [ ]:
query = "python function definition"
unstructured_result = rag_unstructured(query)
structured_result = rag(query)

print("Structured response:\n")
print(structured_result["answer"].model_dump_json(indent=2))

print(f"\nQ5 unstructured input tokens: {unstructured_result['input_tokens']}")
print(f"Q6 structured input tokens: {structured_result['input_tokens']}")
print(f"Unstructured output tokens: {unstructured_result['output_tokens']}")
print(f"Structured output tokens: {structured_result['output_tokens']}")

extra_input_tokens = (
    structured_result["input_tokens"] - unstructured_result["input_tokens"]
)
print(f"\nMORE input tokens with structured output: {extra_input_tokens}")